In [1]:
#%pip install pandas==2.2.3
#%pip install git+https://github.com/pydata/pandas-datareader.git
#%pip install yfinance numpy matplotlib
#%pip install fredapi
#%pip install pyarrow
#%pip install scikit-learn

In [2]:
#Imports necesarios para mdescargar los datos y manejar el dataset
import pandas as pd #Importamos pandas para manejar los dataframes
import numpy as np #Importamos numpy para manejar arrays y realizar operaciones matemáticas
import matplotlib.pyplot as plt
import yfinance as yf #Importamos yfinance para descargar los datos financieros
from pandas_datareader import data as pdr #Importamos pandas_datareader para descargar datos de fuentes financieras
from fredapi import Fred #Importamos fredapi para descargar datos económicos de la Reserva Federal
from sklearn.feature_selection import mutual_info_regression #Importamos mutual_info_regression  para la selección de características
from sklearn.preprocessing import KBinsDiscretizer #Importamos KBinsDiscretizer para discretizar las variables continuas




In [3]:
def imputar_valores_faltantes(dataframe, nombre_columna=None):
    """
    Imputa valores faltantes en una columna de un DataFrame o Serie de pandas. Dicha imputación se realiza utilizando una combinación de tres métodos: interpolación lineal, medias móviles ponderadas y forward/backward fill. 
    Parámetros:
    - dataframe: pd.DataFrame que contiene los datos a imputar.
    - nombre_columna: str, opcional. Nombre de la columna a imputar si se proporciona un DataFrame. 
    Retorna:
    - pd.Series con los valores imputados. 
    """

    if isinstance(dataframe, pd.Series): #Si el input es una Serie, se trabaja directamente con ella
        serie_imputada = dataframe.copy()
    elif isinstance(dataframe, pd.DataFrame): #Si el input es un DataFrame, se selecciona la columna especificada o la única columna disponible
        if nombre_columna is not None: #
            serie_imputada = dataframe[nombre_columna].copy()
        elif dataframe.shape[1] == 1:
            serie_imputada = dataframe.iloc[:, 0].copy()
        else:
            raise ValueError("Debes especificar 'nombre_columna'")
    else:
        raise TypeError("Input debe ser DataFrame o Series")

    serie_imputada = pd.to_numeric(serie_imputada, errors='coerce') #Convertimos a numérico, forzando a NaN los valores no convertibles

    valores_cero = (serie_imputada == 0) #Identificamos los valores que son exactamente cero, ya que se considerarán como faltantes
    valores_nan = serie_imputada.isna() #Identificamos los valores que ya son NaN, para tenerlos en cuenta en el proceso de imputación
    total_a_imputar = valores_cero.sum() + valores_nan.sum() #Calculamos el total de valores que necesitan imputación, sumando los ceros y los NaN

    serie_imputada[valores_cero] = np.nan #Reemplazamos los valores cero por NaN para tratarlos como faltantes en el proceso de imputación

    if total_a_imputar == 0: #Si no hay valores a imputar, se devuelve la serie original sin modificaciones
        return serie_imputada

    #Método 1: Interpolación lineal
    metodo1 = serie_imputada.interpolate(method='linear', limit_direction='both') #La interpolación lineal se realiza utilizando el método 'linear' de pandas, con la opción 'limit_direction' establecida en 'both' para permitir la interpolación tanto hacia adelante como hacia atrás.

    #Método 2: Medias móviles ponderadas
    media_3m = serie_imputada.rolling(window=3, center=True, min_periods=1).mean() #Calculamos la media móvil de 3 meses utilizando el método 'rolling' de pandas, con una ventana de 3 y centrada en el valor actual. La opción 'min_periods=1' permite calcular la media incluso si hay menos de 3 valores disponibles.
    media_6m = serie_imputada.rolling(window=6, center=True, min_periods=1).mean() #Calculamos la media móvil de 6 meses de manera similar, pero con una ventana de 6. Esta media proporciona una visión más suavizada de la serie, capturando tendencias a más largo plazo.
    metodo2 = media_3m * 0.7 + media_6m * 0.3 #Combinamos las dos medias móviles utilizando pesos de 0.7 para la media de 3 meses y 0.3 para la media de 6 meses, dando más importancia a la media más reciente.

    #Método 3: Forward/backward fill
    ffill = serie_imputada.ffill() #El método forward fill (ffill) rellena los valores faltantes con el último valor no nulo conocido hacia adelante.
    bfill = serie_imputada.bfill() #El método backward fill (bfill) rellena los valores faltantes con el primer valor no nulo conocido hacia atrás.
    metodo3 = ffill.fillna(bfill) #Combinamos ambos métodos de relleno, utilizando forward fill primero y luego backward fill para cubrir cualquier valor que no haya sido imputado por el forward fill.

    mask = serie_imputada.isna() #Creamos una máscara booleana que identifica los índices de los valores que aún son NaN después de aplicar los métodos anteriores, para proceder a imputarlos utilizando una combinación ponderada de los tres métodos.
    valores_nulos = np.where(mask)[0] #Obtenemos los índices de los valores que necesitan imputación, utilizando np.where para encontrar las posiciones donde la máscara es True.

    #Imputamos los valores faltantes utilizando una combinación ponderada de los tres métodos, asignando pesos de 0.5 a la interpolación lineal, 0.3 a las medias móviles ponderadas y 0.2 al forward/backward fill. 
    for valor in valores_nulos:
        valores = []
        pesos = []

        if valor < len(metodo1) and not pd.isna(metodo1.iloc[valor]): #Verificamos que el índice del valor a imputar esté dentro del rango de la serie y que el resultado del método 1 no sea NaN antes de agregarlo a la lista de valores y pesos para la imputación.
            valores.append(metodo1.iloc[valor])
            pesos.append(0.50)

        if valor < len(metodo2) and not pd.isna(metodo2.iloc[valor]): #Realizamos la misma verificación para el método 2, asegurándonos de que el índice sea válido y que el valor no sea NaN antes de incluirlo en la combinación ponderada.
            valores.append(metodo2.iloc[valor])
            pesos.append(0.30)

        if valor < len(metodo3) and not pd.isna(metodo3.iloc[valor]): #De manera similar, verificamos el método 3 para asegurarnos de que el índice sea válido y que el valor no sea NaN antes de agregarlo a la lista de valores y pesos para la imputación.
            valores.append(metodo3.iloc[valor])
            pesos.append(0.20)
  
        if valores: #Si hay al menos un valor válido para la imputación, calculamos el valor imputado utilizando una media ponderada de los valores disponibles, normalizando los pesos para que sumen 1 y asignando el resultado a la posición correspondiente en la serie imputada.
            pesos_normalizados = np.array(pesos) / np.sum(pesos)
            valor_imputado = np.average(valores, weights=pesos_normalizados)
            serie_imputada.iloc[valor] = valor_imputado

    if serie_imputada.isna().any(): #Después de aplicar la combinación ponderada de los tres métodos, verificamos si aún quedan valores NaN en la serie imputada. Si es así, aplicamos una interpolación lineal adicional para intentar imputar cualquier valor restante, utilizando el método 'linear' con 'limit_direction' establecido en 'both' para permitir la interpolación en ambas direcciones.
        serie_imputada = serie_imputada.interpolate(method='linear', limit_direction='both')

    if serie_imputada.isna().any(): #Si después de la interpolación lineal aún quedan valores NaN, aplicamos un relleno hacia adelante (ffill) seguido de un relleno hacia atrás (bfill) para asegurarnos de que no queden valores faltantes en la serie imputada, utilizando el método 'ffill' para rellenar hacia adelante y luego 'bfill' para cubrir cualquier valor que no haya sido imputado por el forward fill.
        serie_imputada = serie_imputada.ffill()
        serie_imputada = serie_imputada.bfill()

    return serie_imputada

In [4]:
def seleccion_variables(df, target='target', 
                        umbral_mi_percentil=50,
                        umbral_correlacion=0.80,
                        n_bins=10,
                        random_state=42):
    
    '''Realiza la selección de variables utilizando información mutua, eliminando variables altamente correlacionadas y aplicando un umbral basado en percentiles. Además, permite forzar la inclusión de ciertas variables clave.
    Primero ordenaremos las variables por su información mutua con el target, luego eliminaremos aquellas que estén altamente correlacionadas entre sí, y finalmente aplicaremos un umbral basado en percentiles para seleccionar las variables más relevantes.   
    Parámetros:
    - df: pd.DataFrame que contiene las variables predictoras y el target.
    - target: str, nombre de la columna objetivo.
    - umbral_mi_percentil: int, percentil para establecer el umbral de información mutua.
    - umbral_correlacion: float, umbral para eliminar variables altamente correlacionadas.
    - n_bins: int, número de bins para discretizar las variables continuas al calcular la información mutua.
    - random_state: int, semilla para reproducibilidad.
    Retorna:
    - pd.DataFrame con las variables seleccionadas y el target.
    '''

    variables_forzadas = [
        'sp500_return', 'vix', 'tipos_fed', 
        'curva10Y2Y', 'inflacion_usa', 'activo_return'
    ] #Lista de variables clave que se desea incluir en el modelo final, independientemente de su información mutua.
    

    X = df.select_dtypes(include=[np.number]).drop(columns=[target]) #Seleccionamos solo las columnas numéricas del DataFrame para trabajar con variables predictoras, y eliminamos la columna objetivo (target) para evitar que se incluya en el cálculo de la información mutua.
    y = df[target] #Asignamos la columna objetivo a la variable y para su uso en el cálculo de la información mutua y la selección de variables.
    
    # Discretizar X
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform') #Creamos un discretizador que dividirá las variables continuas en n_bins utilizando una estrategia uniforme, y codificará los bins resultantes como números ordinales.
    X_discrete = discretizer.fit_transform(X) #Aplicamos el discretizador a las variables predictoras X, transformándolas en variables discretas que pueden ser utilizadas para calcular la información mutua de manera similar a como lo hace R.
    X_discrete = pd.DataFrame(X_discrete, columns=X.columns, index=X.index) #Convertimos el resultado del discretizador en un DataFrame de pandas, manteniendo los mismos nombres de columnas e índices que el DataFrame original X para facilitar la interpretación de los resultados.
    
    # Discretizar y 
    y_discrete = pd.cut(y, bins=n_bins, labels=False)
    
    # Calcular MI con variables discretas
    mi_scores = mutual_info_regression(X_discrete, y_discrete, random_state=random_state) #Calculamos la información mutua entre las variables discretas de X y el target discretizado utilizando mutual_info_regression de scikit-learn, est devuelve un array con los scores de información mutua para cada variable en X.
    
    mi_df = pd.DataFrame({
        'variable': X.columns,
        'mi_score': mi_scores
    }).sort_values('mi_score', ascending=False) #Creamos un DataFrame que contiene los nombres de las variables y sus correspondientes scores de información mutua, y ordenamos el DataFrame de mayor a menor según la información mutua para identificar las variables más relevantes.
    
    for i, row in mi_df.head(15).iterrows():
        print(f"   {row['variable']:25s} {row['mi_score']:.6f}")
    
    
        
    mi_dict = dict(zip(mi_df['variable'], mi_df['mi_score'])) #Creamos un diccionario que mapea cada variable con su score de información mutua, lo que facilitará la consulta de los scores durante el proceso de selección de variables y la impresión de los resultados finales.
    vars_ordenadas = mi_df['variable'].tolist()
    
    variables_sin_corr = []
    matriz_corr = X.corr().abs() #Calculamos la matriz de correlación absoluta entre las variables predictoras para identificar aquellas que están altamente correlacionadas entre sí.
    
    for var in vars_ordenadas: #Iteramos sobre las variables ordenadas por información mutua, comenzando por la más relevante, para seleccionar aquellas que no estén altamente correlacionadas con las ya seleccionadas.
        es_redundante = False
        
        for selected in variables_sin_corr: #Para cada variable en el orden de importancia, verificamos su correlación con las variables que ya han sido seleccionadas y no tienen correlación. Si la correlación con alguna de las variables seleccionadas supera el umbral establecido, marcamos la variable actual como redundante y no la incluimos en la lista de variables sin correlación.
            corr_val = matriz_corr.loc[var, selected]
            if corr_val > umbral_correlacion:
                es_redundante = True
                break
        
        if not es_redundante: #En caso de que la variable no sea redundante, entonces la añadimos a la lista de variables sin correlación.
            variables_sin_corr.append(var)
    
    
    n_vars = len(variables_sin_corr) #Calculamos el número de variables que no están altamente correlacionadas entre sí, lo que nos ayudará a saber cuántas variables tenemos disponibles para aplicar el umbral basado en percentiles de información mutua.   
    umbral_index = int(np.ceil(n_vars * umbral_mi_percentil / 100)) #Calculamos el umbral de selección para la información mutua basado en el percentil especificado.
    
    vars_seleccionadas = variables_sin_corr[:umbral_index] #Seleccionamos las variables que cumplen con el umbral de información mutua, tomando las primeras variables sin correlación hasta el índice calculado por el umbral.
    

    for var in variables_forzadas: #Finalmente, verificamos si las variables clave que se desean forzar en el modelo están presentes en el DataFrame original y no han sido ya seleccionadas por el proceso de selección. Si cumplen estas condiciones, las añadimos a la lista de variables seleccionadas, asegurándonos de que estén incluidas en el modelo final independientemente de su información mutua.
        if var in X.columns and var not in vars_seleccionadas:
            vars_seleccionadas.append(var)
    
    print(f"\nVariables seleccionadas:")
    for i, var in enumerate(vars_seleccionadas, 1):
        mi_val = mi_dict.get(var, 0)
        es_forzada = " [FORZADA]" if var in variables_forzadas else ""
        print(f"   {i:2d}. {var:30s} (MI: {mi_val:.6f}){es_forzada}")
    
    # ============================================
    # 8. CREAR DATASET FINAL
    # ============================================
    dataset_final = df[vars_seleccionadas + [target]].copy()
    
    return dataset_final

In [5]:
dataset = pd.DataFrame() #dataset global

inicio = '2001-12-01' #fecha en la que empezamos a descargar datos
fin = '2026-03-01' #fecga en la que dejamos de descargar datos

sp500 = yf.download('^GSPC', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierredel sp500
sp500 = sp500.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
sp500["sp500_return"] = sp500.pct_change() #Calculamos el retorno mensual del sp500 y lo guardamos en una nueva columna
sp500 = sp500.dropna() #Eliminamos los valores nulos
sp500 = sp500[["sp500_return"]]
#sp500.info()

sp500 = sp500.reset_index()
dataset = sp500 #Asignamos el dataset global al dataset del sp500, que es el que vamos a ir ampliando con las demás variables
dataset.head()

[*********************100%***********************]  1 of 1 completed


Ticker,Date,sp500_return
0,2002-01-31,-0.015574
1,2002-02-28,-0.020766
2,2002-03-31,0.036739
3,2002-04-30,-0.061418
4,2002-05-31,-0.009081


In [6]:
eurodollar = yf.download('EURUSD=X', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del eurodolar
#eurodollar.info()
eurodollar = eurodollar.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes
eurodollar = eurodollar.rename(columns={"Close": "EUR/USD"})
eurodollar = eurodollar.reset_index()

dataset = pd.merge(dataset,eurodollar, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del eurodolar, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets

dataset.head()

[*********************100%***********************]  1 of 1 completed


Ticker,Date,sp500_return,EURUSD=X
0,2003-12-31,0.050766,1.259002
1,2004-01-31,0.017276,1.245206
2,2004-02-29,0.012209,1.253007
3,2004-03-31,-0.016359,1.231300
4,2004-04-30,-0.016791,1.198294


In [7]:
vix = yf.download('^VIX', start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del vix
vix.info()
vix = vix.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el vix ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
vix = vix.rename(columns={"Close":"vix"}) #Renombramos la columna de cierre a "vix" para que tenga un nombre más descriptivo y consistente con el resto de variables en el dataset.
vix = vix.reset_index()
#vix.head()

dataset = pd.merge(dataset, vix, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del vix, uniendo por la columna "Date" y quedándonos solo con las filas que tengan fecha en ambos datasets
dataset.head()

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-01 to 2026-02-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ^VIX    291 non-null    float64
dtypes: float64(1)
memory usage: 4.5 KB


Ticker,Date,sp500_return,EURUSD=X,^VIX
0,2003-12-31,0.050766,1.259002,18.309999
1,2004-01-31,0.017276,1.245206,16.629999
2,2004-02-29,0.012209,1.253007,14.550000
3,2004-03-31,-0.016359,1.231300,16.740000
4,2004-04-30,-0.016791,1.198294,17.190001


In [8]:
petroleo = yf.download('CL=F', start=inicio, end=fin, interval='1mo') 
petroleo.info()
petroleo.columns = petroleo.columns.get_level_values(0) #El dataset del petroleo tiene un multiindex en las columnas, con el nombre del ticker y el nombre de la variable, por lo que tenemos que eliminar el nivel del ticker para quedarnos solo con el nombre de la variable
petroleo = petroleo.resample("ME").last() 
petroleo = petroleo[['Close']].copy() #Seleccionamos solo la columna de cierre, que es la que nos interesa para calcular el retorno del petroleo.
petroleo['Close'] = imputar_valores_faltantes(petroleo, 'Close') #Imputamos los valores faltantes en la columna de cierre del petroleo utilizando la función de imputación definida anteriormente, para asegurarnos de que no haya valores nulos antes de calcular el retorno del petroleo.
petroleo["petroleo_return"] = petroleo["Close"].pct_change() #Calculamos el retorno del petroleo utilizando la función pct_change() y guardamos el resultado en una nueva columna llamada "petroleo_return".
petroleo = petroleo[["petroleo_return"]] #Nos quedamos solo con la columna de retorno del petroleo
petroleo = petroleo.reset_index()

dataset = pd.merge(dataset, petroleo, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 249 entries, 2001-12-01 to 2026-01-01
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, CL=F)   249 non-null    float64
 1   (High, CL=F)    249 non-null    float64
 2   (Low, CL=F)     249 non-null    float64
 3   (Open, CL=F)    249 non-null    float64
 4   (Volume, CL=F)  249 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 11.7 KB


In [9]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             266 non-null    datetime64[ns]
 1   sp500_return     266 non-null    float64       
 2   EURUSD=X         266 non-null    float64       
 3   ^VIX             266 non-null    float64       
 4   petroleo_return  266 non-null    float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 10.5 KB


In [10]:
oro = yf.download('GC=F', start=inicio, end=fin, interval='1mo') #Descargamos los datos de cierre del oro
oro.info()  
oro.columns = oro.columns.get_level_values(0) 
oro = oro.resample("ME").last() 
oro = oro[['Close']].copy()
oro['Close'] = imputar_valores_faltantes(oro, 'Close')
oro["oro_return"] = oro["Close"].pct_change()
oro = oro[["oro_return"]]
oro = oro.reset_index()

dataset = pd.merge(dataset, oro, on="Date", how="inner")

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 249 entries, 2001-12-01 to 2026-01-01
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, GC=F)   249 non-null    float64
 1   (High, GC=F)    249 non-null    float64
 2   (Low, GC=F)     249 non-null    float64
 3   (Open, GC=F)    249 non-null    float64
 4   (Volume, GC=F)  249 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 11.7 KB


In [11]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             266 non-null    datetime64[ns]
 1   sp500_return     266 non-null    float64       
 2   EURUSD=X         266 non-null    float64       
 3   ^VIX             266 non-null    float64       
 4   petroleo_return  266 non-null    float64       
 5   oro_return       266 non-null    float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 12.6 KB


In [12]:
fred = Fred(api_key="af59516007957e0c89174b4d426736ca") #Creamos un objeto de la clase Fred utilizando la clave de API proporcionada, lo que nos permitirá acceder a los datos económicos disponibles en la base de datos de la Reserva Federal a través de la biblioteca fredapi.

tipos_fed = fred.get_series("FEDFUNDS", observation_start=inicio, observation_end=fin) #Descargamos la serie de los tipos de interés de la Reserva Federal
tipos_fed = tipos_fed.to_frame() #Convertimos la serie en un DataFrame

tipos_fed.info()
tipos_fed = tipos_fed.resample("ME").last()
tipos_fed = tipos_fed.rename(columns={tipos_fed.columns[0]: "tipos_fed"}) #Renombramos la columna de tipos de interés a "tipos_fed" para que tenga un nombre más descriptivo y consistente con el resto de variables en el dataset.
tipos_fed = tipos_fed.reset_index()
tipos_fed = tipos_fed.rename(columns={"index":"Date"}) #Renombramos la columna de índice a "Date" para poder hacer el merge con el dataset global.

dataset = pd.merge(dataset, tipos_fed, on="Date", how="inner")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       292 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


In [13]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00


In [14]:
ipc_usa = fred.get_series("CPIAUCSL", observation_start=inicio, observation_end=fin) #Descargamos los datos del IPC de Estados Unidos desde la base de datos de FRED, utilizando el código CPIAUCSL que corresponde al IPC de Estados Unidos, y especificando el rango de fechas que queremos descargar
ipc_usa = ipc_usa.to_frame() 

ipc_usa.info()
ipc_usa = ipc_usa.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_usa["inflacion_usa"] = ipc_usa.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de Estados Unidos usando iloc para acceder a la primera columna (evita problemas de nombres)
ipc_usa = ipc_usa[["inflacion_usa"]]
ipc_usa = ipc_usa.reset_index()
ipc_usa = ipc_usa.rename(columns={"index": "Date"}) #Cambiamos el nombre de la columna de índice a "Date" para poder hacer el merge con el dataset global.

dataset = pd.merge(dataset, ipc_usa, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de Estados Unidos, uniendo por la columna Date

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       291 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


C:\Users\34620\AppData\Local\Temp\ipykernel_12848\2969586754.py:6: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ipc_usa["inflacion_usa"] = ipc_usa.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de Estados Unidos usando iloc para acceder a la primera columna (evita problemas de nombres)


In [15]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926


In [16]:
ipc_eurozona = fred.get_series("CP0000EZ19M086NEST", observation_start=inicio, observation_end=fin)
ipc_eurozona = ipc_eurozona.to_frame()

ipc_eurozona.info()
ipc_eurozona = ipc_eurozona.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
ipc_eurozona["inflation_eu"] = ipc_eurozona.iloc[:, 0].pct_change(12) #Calculamos la inflación interanual de la eurozona usando iloc para acceder a la primera columna 
ipc_eurozona = ipc_eurozona[["inflation_eu"]]
ipc_eurozona = ipc_eurozona.reset_index()
ipc_eurozona = ipc_eurozona.rename(columns={"index": "Date"}) #Cambiamos el nombre de la columna de índice a "Date" para poder hacer el merge con el dataset global.

dataset = pd.merge(dataset, ipc_eurozona, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       292 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


In [17]:
dataset.head()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu
0,2003-12-31,0.050766,1.259002,18.309999,0.069385,0.047631,0.98,0.020352,0.020225
1,2004-01-31,0.017276,1.245206,16.629999,0.016298,-0.032475,1.00,0.020263,0.018468
2,2004-02-29,0.012209,1.253007,14.550000,0.031217,0.022960,1.01,0.016885,0.016795
3,2004-03-31,-0.016359,1.231300,16.740000,0.049243,0.038561,1.00,0.017401,0.017335
4,2004-04-30,-0.016791,1.198294,17.190001,0.045302,-0.094313,1.00,0.022926,0.020959


In [18]:
desempleoUSA = fred.get_series("UNRATE", observation_start=inicio, observation_end=fin)
desempleoUSA = desempleoUSA.to_frame()

desempleoUSA.info()
desempleoUSA = desempleoUSA.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el desempleo de Estados Unidos ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
desempleoUSA = desempleoUSA.rename(columns={desempleoUSA.columns[0]: "desempleo"})
desempleoUSA = desempleoUSA.reset_index()
desempleoUSA = desempleoUSA.rename(columns={"index": "Date"}) #Cambiamos el nombre de la columna de índice a "Date" para poder hacer el merge con el dataset global.

dataset = pd.merge(dataset, desempleoUSA, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del desempleo de Estados Unidos, uniendo por la columna Date

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 292 entries, 2001-12-01 to 2026-03-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       291 non-null    float64
dtypes: float64(1)
memory usage: 4.6 KB


In [19]:
dataset.tail()

,Date,sp500_return,EURUSD=X,^VIX,petroleo_return,oro_return,tipos_fed,inflacion_usa,inflation_eu,desempleo
261,2025-09-30,0.035324,1.173144,16.280001,-0.025621,0.105680,4.22,0.030226,0.022168,4.4
262,2025-10-31,0.022687,1.157247,17.440001,-0.022286,0.036815,4.09,0.027291,0.020774,NaN
263,2025-11-30,0.001300,1.159689,16.350000,-0.039849,0.059289,3.88,0.026964,0.021147,4.5
264,2025-12-31,-0.000524,1.174729,14.950000,-0.019300,0.025437,3.72,0.026533,0.019350,4.4
265,2026-01-31,0.013663,1.185536,17.440001,0.135667,0.089768,3.64,0.023912,0.016357,4.3


In [20]:
tipos_ecb = fred.get_series("ECBDFR") #Descargamos los datos de los tipos de interés de la ECB desde la base de datos de FRED
tipos_ecb = tipos_ecb.to_frame() 

tipos_ecb.info()
tipos_ecb = tipos_ecb.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los tipos de interés de la ECB ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables
tipos_ecb = tipos_ecb.rename(columns={tipos_ecb.columns[0]: "tipos_ecb"})
tipos_ecb = tipos_ecb.reset_index()
tipos_ecb = tipos_ecb.rename(columns={"index":"Date"}) #Renombramos la columna index a Date


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 10000 entries, 1999-01-01 to 2026-05-18
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       10000 non-null  float64
dtypes: float64(1)
memory usage: 156.2 KB


In [21]:
dataset = pd.merge(dataset, tipos_ecb, on="Date", how="inner")

In [22]:
curva10Y2 = fred.get_series("T10Y2Y") #Descargamos los datos de la curva de tipos de interés a 10 años menos la curva de tipos de interés a 2 años desde la base de datos de FRED
curva10Y2 = curva10Y2.to_frame()

curva10Y2.info()
curva10Y2 = curva10Y2.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque el IPC de la eurozona ya se publica con datos mensuales, pero lo hacemos por consistencia con el resto de variables
curva10Y2 = curva10Y2.rename(columns={curva10Y2.columns[0]: "curva10Y2Y"})
curva10Y2 = curva10Y2.reset_index()
curva10Y2 = curva10Y2.rename(columns={"index":"Date"}) #Renombramos la columna index a Date para poder hacer el merge con el dataset global.


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13035 entries, 1976-06-01 to 2026-05-18
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       12487 non-null  float64
dtypes: float64(1)
memory usage: 203.7 KB


In [23]:
dataset = pd.merge(dataset, curva10Y2, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del IPC de la eurozona, uniendo por la columna Date 

In [24]:
dataset = dataset.rename(columns={"0_x":"tipos_ecb"}) #Renombramos la columna 0_x a tipos_ecb
dataset = dataset.rename(columns={"0_y":"curva10Y2Y"}) #Renombramos la columna 0_y a curva10Y2Y
dataset.tail()
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             266 non-null    datetime64[ns]
 1   sp500_return     266 non-null    float64       
 2   EURUSD=X         266 non-null    float64       
 3   ^VIX             266 non-null    float64       
 4   petroleo_return  266 non-null    float64       
 5   oro_return       266 non-null    float64       
 6   tipos_fed        266 non-null    float64       
 7   inflacion_usa    266 non-null    float64       
 8   inflation_eu     266 non-null    float64       
 9   desempleo        265 non-null    float64       
 10  tipos_ecb        266 non-null    float64       
 11  curva10Y2Y       266 non-null    float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 25.1 KB


In [ ]:
activo = 'NVDA'

datos_activo = yf.download(activo, start=inicio, end=fin, interval='1mo')['Close'] #Descargamos los datos de cierre del activo que queramos analizar
datos_activo.info()
datos_activo = datos_activo.resample("ME").last() #Cogemos el valor de cierre del último día de cada mes, aunque en este caso no es necesario porque los datos del activo ya se publican con datos mensuales, pero lo hacemos por consistencia con el resto de variables    
#datos_activo = datos_activo.to_frame()
datos_activo = datos_activo.rename(columns={activo:"precio"})
datos_activo["activo_return"] = datos_activo["precio"].pct_change() #Calculamos el retorno mensual del activo y lo guardamos en una nueva columna

datos_activo['media_movil_3m'] = datos_activo["precio"].rolling(3).mean() #Calculamos la media móvil de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_6m'] = datos_activo["precio"].rolling(6).mean() #Calculamos la media móvil de 6 meses del precio del activo y la guardamos en una nueva columna
datos_activo['media_movil_12m'] = datos_activo["precio"].rolling(12).mean() #Calculamos la media móvil de 12 meses del precio del activo y la guardamos en una nueva columna

datos_activo['volatilidad_3m'] = datos_activo["activo_return"].rolling(3).std() #Calculamos la volatilidad de 3 meses del precio del activo y la guardamos en una nueva columna
datos_activo['volatilidad_6m'] = datos_activo["activo_return"].rolling(6).std() #Calculamos la volatilidad de 6 meses del precio del activo y la guardamos en una nueva columna

datos_activo['momentum_3m'] = datos_activo["precio"].rolling(3).sum() #Calculamos el momentum de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_6m'] = datos_activo["precio"].rolling(6).sum() #Calculamos el momentum de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['momentum_12m'] = datos_activo["precio"].rolling(12).sum() #Calculamos el momentum de 12 meses del precio del activo y lo guardamos en una nueva columna

datos_activo['lag_1'] = datos_activo["activo_return"].shift(1) #Calculamos el lag de 1 mes del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_2'] = datos_activo["activo_return"].shift(2) #Calculamos el lag de 2 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_3'] = datos_activo["activo_return"].shift(3) #Calculamos el lag de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_6'] = datos_activo["activo_return"].shift(6) #Calculamos el lag de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_12'] = datos_activo["activo_return"].shift(12) #Calculamos el lag de 12 meses del precio del activo y lo guardamos en una nueva columna
datos_activo.info()

datos_activo = datos_activo.reset_index()

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-01 to 2026-02-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    291 non-null    float64
dtypes: float64(1)
memory usage: 4.5 KB
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 291 entries, 2001-12-31 to 2026-02-28
Freq: ME
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   precio           291 non-null    float64
 1   activo_return    290 non-null    float64
 2   media_movil_3m   289 non-null    float64
 3   media_movil_6m   286 non-null    float64
 4   media_movil_12m  280 non-null    float64
 5   volatilidad_3m   288 non-null    float64
 6   volatilidad_6m   285 non-null    float64
 7   momentum_3m      289 non-null    float64
 8   momentum_6m      286 non-null    float64
 9   momentum_12m     280 non-null    float64
 10  lag_1            289 non-null    flo

In [26]:
dataset = pd.merge(dataset, datos_activo, on="Date", how="inner") #Hacemos un merge entre el dataset global y el dataset del activo, uniendo por la columna Date

In [27]:
dataset = dataset.rename(columns={
    "EURUSD=X": "eurusd",
    "^VIX": "vix"
}) #Renombramos las columnas del eurodolar y del vix para que tengan nombres más cortos y fáciles de manejar

#dataset.info()
#dataset.iloc[258:]
#dataset.tail()

#A continuación se imputan los valores faltantes en las columnas que lo necesiten utilizando la función de imputación definida anteriormente, para asegurarnos de que no haya valores nulos en el dataset antes de proceder con el análisis y la selección de variables.
dataset['desempleo'] = imputar_valores_faltantes(dataset, 'desempleo')
dataset['activo_return'] = imputar_valores_faltantes(dataset, 'activo_return')
dataset['media_movil_3m'] = imputar_valores_faltantes(dataset, 'media_movil_3m')
dataset['media_movil_6m'] = imputar_valores_faltantes(dataset, 'media_movil_6m')
dataset['media_movil_12m'] = imputar_valores_faltantes(dataset, 'media_movil_12m')
dataset['media_movil_3m'] = imputar_valores_faltantes(dataset, 'media_movil_3m')
dataset['volatilidad_3m'] = imputar_valores_faltantes(dataset, 'volatilidad_3m')
dataset['volatilidad_6m'] = imputar_valores_faltantes(dataset, 'volatilidad_6m')
dataset['momentum_3m'] = imputar_valores_faltantes(dataset, 'momentum_3m')
dataset['momentum_6m'] = imputar_valores_faltantes(dataset, 'momentum_6m')
dataset['momentum_12m'] = imputar_valores_faltantes(dataset, 'momentum_12m')
dataset['lag_1'] = imputar_valores_faltantes(dataset, 'lag_1')
dataset['lag_2'] = imputar_valores_faltantes(dataset, 'lag_2')
dataset['lag_3'] = imputar_valores_faltantes(dataset, 'lag_3')
dataset['lag_6'] = imputar_valores_faltantes(dataset, 'lag_6')
dataset['lag_12'] = imputar_valores_faltantes(dataset, 'lag_12')

dataset['desempleo'] = dataset['desempleo'].round(2)
dataset.isnull().sum()
#dataset.iloc[258:] 

dataset["target"] = dataset["activo_return"].shift(-1) #Creamos la columna target, que es el retorno del activo en el mes siguiente
dataset['target'] = imputar_valores_faltantes(dataset, 'target')

In [28]:
#dataset['target'].isnull()
#dataset.iloc[:6]
print(dataset.columns)

#Vamos a ver los valores de cada columna, para asi ver posibles anomalias
print(dataset['petroleo_return'].value_counts()) 
print(dataset['oro_return'].value_counts()) 

mask = (dataset['oro_return'] == 0) | (dataset['petroleo_return'] == 0)

#print(dataset[mask].head()) #Mostramos las filas donde el retorno del oro o del petroleo es 0, para ver si hay alguna anomalía en esas fechas
#Una vez arreglado el problema de los 0, vemos que el oro y petroleo return, ya están bien

#print(dataset['petroleo_return'].value_counts()) 
# #dataset['oro_return'].value_counts() 

#dataset[['Date','activo_return', 'target']]


#dataset.dtypes

dataset.sort_values(by="Date", ascending=True, inplace=True) #Ordenamos el dataset por la columna Date para asegurarnos de que los datos están en orden cronológico
dataset.head()

dataset.to_csv("dataset_completo_exp.csv", index=False) #Guardamos el dataset completo en un archivo csv para poder usarlo posteriormente en el análisis de selección de variables y en la construcción del modelo predictivo

Index(['Date', 'sp500_return', 'eurusd', 'vix', 'petroleo_return',
       'oro_return', 'tipos_fed', 'inflacion_usa', 'inflation_eu', 'desempleo',
       'tipos_ecb', 'curva10Y2Y', 'precio', 'activo_return', 'media_movil_3m',
       'media_movil_6m', 'media_movil_12m', 'volatilidad_3m', 'volatilidad_6m',
       'momentum_3m', 'momentum_6m', 'momentum_12m', 'lag_1', 'lag_2', 'lag_3',
       'lag_6', 'lag_12', 'target'],
      dtype='object')
petroleo_return
 0.069385    1
 0.016298    1
 0.031217    1
 0.049243    1
 0.045302    1
            ..
-0.025621    1
-0.022286    1
-0.039849    1
-0.019300    1
 0.135667    1
Name: count, Length: 266, dtype: int64
oro_return
 0.047631    1
-0.032475    1
 0.022960    1
 0.038561    1
-0.094313    1
            ..
 0.105680    1
 0.036815    1
 0.059289    1
 0.025437    1
 0.089768    1
Name: count, Length: 266, dtype: int64


In [29]:
dataset_optmizado = seleccion_variables(dataset, target='target') #Aplicamos la función de selección de variables al dataset completo para obtener un dataset optimizado con las variables más relevantes para predecir el target, lo que nos permitirá construir un modelo predictivo más eficiente y con mejor rendimiento.

variables_seleccionadas = [col for col in dataset_optmizado.columns if col != 'target'] #Creamos una lista de las variables seleccionadas, excluyendo la columna target, para poder trabajar con ellas en el análisis y la construcción del modelo predictivo.
variables_macro_seleccionadas = [v for v in variables_seleccionadas if v in ['tipos_fed', 'inflacion_usa', 'vix', 'desempleo', 'curva10Y2Y']] #Creamos una lista de las variables macroeconómicas seleccionadas, filtrando la lista de variables seleccionadas para incluir solo aquellas que estén en la lista de variables macroeconómicas que nos interesan, lo que nos permitirá enfocarnos en estas variables para el análisis y la construcción del modelo predictivo.

dataset_dl = dataset_optmizado.copy() #Creamos una copia del dataset para usarlo en el modelo de deep learning, asi mantenemos el dataset original por si queremos hacer alguna otra cosa con él
dataset_ml = dataset_optmizado.copy() #Creamos una copia del dataset para usarlo en el modelo de machine learning, asi mantenemos el dataset original por si queremos hacer alguna otra cosa con él

if 'tipos_fed' in variables_macro_seleccionadas: #Si la variable "tipos_fed" está entre las variables macroeconómicas seleccionadas, entonces creamos nuevas columnas en el dataset de machine learning que contienen los lags de "tipos_fed" de 1, 3 y 6 meses hacia adelante.
    dataset_ml['tipos_fed_lag1'] = dataset_ml['tipos_fed'].shift(1)
    dataset_ml['tipos_fed_lag3'] = dataset_ml['tipos_fed'].shift(3)
    dataset_ml['tipos_fed_lag6'] = dataset_ml['tipos_fed'].shift(6)


if 'inflacion_usa' in variables_macro_seleccionadas: #Si la variable "inflacion_usa" está entre las variables macroeconómicas seleccionadas, entonces creamos nuevas columnas en el dataset de machine learning que contienen los lags de "inflacion_usa" de 1, 3 y 6 meses hacia adelante.
    dataset_ml['inflacion_usa_lag1'] = dataset_ml['inflacion_usa'].shift(1)
    dataset_ml['inflacion_usa_lag3'] = dataset_ml['inflacion_usa'].shift(3)
    dataset_ml['inflacion_usa_lag6'] = dataset_ml['inflacion_usa'].shift(6)
   

if 'vix' in variables_macro_seleccionadas: #Si la variable "vix" está entre las variables macroeconómicas seleccionadas, entonces creamos nuevas columnas en el dataset de machine learning que contienen los lags de "vix" de 1, 2 y 3 meses hacia adelante.
    dataset_ml['vix_lag1'] = dataset_ml['vix'].shift(1)
    dataset_ml['vix_lag2'] = dataset_ml['vix'].shift(2)
    dataset_ml['vix_lag3'] = dataset_ml['vix'].shift(3)
   

if 'desempleo' in variables_macro_seleccionadas: #Si la variable "desempleo" está entre las variables macroeconómicas seleccionadas, entonces creamos nuevas columnas en el dataset de machine learning que contienen los lags de "desempleo" de 1 y 3 meses hacia adelante.
    dataset_ml['desempleo_lag1'] = dataset_ml['desempleo'].shift(1)
    dataset_ml['desempleo_lag3'] = dataset_ml['desempleo'].shift(3)
   

if 'curva10Y2Y' in variables_macro_seleccionadas: #Si la variable "curva10Y2Y" está entre las variables macroeconómicas seleccionadas, entonces creamos nuevas columnas en el dataset de machine learning que contienen los lags de "curva10Y2Y" de 1 y 3 meses hacia adelante.
    dataset_ml['curva10Y2Y_lag1'] = dataset_ml['curva10Y2Y'].shift(1)
    dataset_ml['curva10Y2Y_lag3'] = dataset_ml['curva10Y2Y'].shift(3)

for col in dataset_ml.columns:
    if dataset_ml[col].isnull().any():
        dataset_ml[col] = imputar_valores_faltantes(dataset_ml, col)


   media_movil_12m           0.086456
   oro_return                0.080680
   volatilidad_3m            0.069125
   precio                    0.060270
   momentum_3m               0.058116
   vix                       0.055713
   curva10Y2Y                0.038724
   momentum_12m              0.037458
   media_movil_6m            0.035243
   inflacion_usa             0.034894
   media_movil_3m            0.025724
   lag_3                     0.022091
   lag_12                    0.018851
   lag_2                     0.009515
   momentum_6m               0.008335

Variables seleccionadas:
    1. media_movil_12m                (MI: 0.086456)
    2. oro_return                     (MI: 0.080680)
    3. volatilidad_3m                 (MI: 0.069125)
    4. vix                            (MI: 0.055713) [FORZADA]
    5. curva10Y2Y                     (MI: 0.038724) [FORZADA]
    6. inflacion_usa                  (MI: 0.034894) [FORZADA]
    7. lag_3                          (MI: 0.022091)
   

In [30]:
#print(dataset_ml.info())
#print(dataset_ml.isnull().sum())

In [31]:
#dataset_ml.head()

In [32]:
dataset.head()
dataset_ml.columns

Index(['media_movil_12m', 'oro_return', 'volatilidad_3m', 'vix', 'curva10Y2Y',
       'inflacion_usa', 'lag_3', 'lag_12', 'lag_2', 'sp500_return',
       'tipos_fed', 'activo_return', 'target', 'tipos_fed_lag1',
       'tipos_fed_lag3', 'tipos_fed_lag6', 'inflacion_usa_lag1',
       'inflacion_usa_lag3', 'inflacion_usa_lag6', 'vix_lag1', 'vix_lag2',
       'vix_lag3', 'curva10Y2Y_lag1', 'curva10Y2Y_lag3'],
      dtype='object')

In [33]:
dataset_dl.to_csv('dataset_dl_exp.csv', index=False)
dataset_ml.to_csv('dataset_ml_exp.csv', index=False)